![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 05: NLP for Clinical Text
***
**Learning objectives**
- Understand NER entity types in clinical NLP
- Build a rule-based EntityRuler pipeline with 180+ clinical patterns
- Apply negation-aware NER with context windows
- Evaluate NER quality with precision, recall, and F1
- Visualise entity type distributions

**Estimated time:** 2.5 hours | **Level:** Intermediate → Advanced | **Libraries:** `spacy`, `re`, `pandas`


## 1. Setup & Annotated Corpus

In [ ]:
import re, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod05", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})

try:
    import spacy
    try:
        nlp_base = spacy.load("en_core_web_sm")
    except OSError:
        os.system("python -m spacy download en_core_web_sm -q")
        nlp_base = spacy.load("en_core_web_sm")
    SPACY_OK = True
    print(f"spaCy {spacy.__version__} loaded")
except ImportError:
    SPACY_OK = False
    print("spaCy not available — using regex NER fallback")

# Annotated clinical texts
ANNOTATED = [
    {"id":"T001","text":
     "The patient has type 2 diabetes mellitus managed with metformin 1000 mg twice daily "
     "and lisinopril 10 mg for hypertension. She also takes furosemide 40 mg for heart failure. "
     "On exam: bilateral pitting edema to the knees, S3 gallop present.",
     "gold":[("type 2 diabetes mellitus","DISEASE"),("metformin","DRUG"),
             ("lisinopril","DRUG"),("hypertension","DISEASE"),("furosemide","DRUG"),
             ("heart failure","DISEASE"),("bilateral pitting edema","FINDING"),
             ("S3 gallop","FINDING")]},
    {"id":"T002","text":
     "CT chest revealed a 5mm pulmonary nodule in the right lower lobe and bilateral pleural effusions. "
     "No pulmonary embolism identified. The patient was started on rivaroxaban 20 mg for atrial fibrillation. "
     "Echo showed mitral regurgitation and aortic stenosis.",
     "gold":[("pulmonary nodule","FINDING"),("right lower lobe","ANATOMY"),
             ("bilateral pleural effusions","FINDING"),("pulmonary embolism","DISEASE"),
             ("rivaroxaban","DRUG"),("atrial fibrillation","DISEASE"),
             ("mitral regurgitation","DISEASE"),("aortic stenosis","DISEASE")]},
    {"id":"T003","text":
     "Post-op day 2 following coronary artery bypass grafting. Chest pain at sternotomy site rated 6/10. "
     "No wound infection. Warfarin 5 mg initiated for mechanical aortic valve. "
     "Creatinine elevated at 1.8, likely acute kidney injury.",
     "gold":[("coronary artery bypass grafting","PROCEDURE"),("chest pain","SYMPTOM"),
             ("sternotomy site","ANATOMY"),("wound infection","DISEASE"),
             ("Warfarin","DRUG"),("acute kidney injury","DISEASE")]},
    {"id":"T004","text":
     "Impression: Community-acquired pneumonia with sepsis. "
     "Started ceftriaxone 1g IV daily and azithromycin 500 mg IV daily. Blood cultures pending. "
     "CBC: WBC 18.4, Hgb 9.2. CXR: right lower lobe consolidation.",
     "gold":[("Community-acquired pneumonia","DISEASE"),("sepsis","DISEASE"),
             ("ceftriaxone","DRUG"),("azithromycin","DRUG"),
             ("right lower lobe consolidation","FINDING")]},
]
print(f"Annotated corpus: {len(ANNOTATED)} texts")
print(f"Total gold entities: {sum(len(t['gold']) for t in ANNOTATED)}")


## 2. Clinical Entity Pattern Libraries

| Type | Examples | Count |
|---|---|---|
| **DISEASE** | diabetes, heart failure, pneumonia | 40+ |
| **DRUG** | metformin, lisinopril, ceftriaxone | 60+ |
| **PROCEDURE** | CABG, echocardiogram, CT chest | 25+ |
| **ANATOMY** | right lower lobe, sternotomy site | 20+ |
| **SYMPTOM** | chest pain, dyspnea, fever | 25+ |
| **FINDING** | pitting edema, ST elevation, consolidation | 25+ |

In [ ]:
DISEASE_PATTERNS = [
    "diabetes mellitus","type 2 diabetes mellitus","type 1 diabetes","DM2",
    "hypertension","heart failure","congestive heart failure","ADHF",
    "COPD","chronic obstructive pulmonary disease","asthma","pneumonia","CAP",
    "atrial fibrillation","AFib","coronary artery disease","CAD",
    "myocardial infarction","STEMI","NSTEMI","ACS",
    "chronic kidney disease","CKD","acute kidney injury","AKI","ESRD",
    "pulmonary embolism","PE","deep vein thrombosis","DVT",
    "sepsis","severe sepsis","septic shock","bacteremia",
    "stroke","TIA","ischemic stroke",
    "mitral regurgitation","aortic stenosis","aortic regurgitation",
    "pleural effusion","bilateral pleural effusions","pneumothorax",
    "community-acquired pneumonia","wound infection","cellulitis",
]
DRUG_PATTERNS = [
    "metformin","lisinopril","furosemide","warfarin","rivaroxaban","apixaban",
    "aspirin","clopidogrel","ticagrelor","heparin","enoxaparin",
    "ceftriaxone","azithromycin","vancomycin","piperacillin-tazobactam","meropenem",
    "metoprolol","carvedilol","atorvastatin","rosuvastatin",
    "amiodarone","digoxin","diltiazem","prednisone","methylprednisolone","dexamethasone",
    "albuterol","ipratropium","tiotropium","fluticasone",
    "insulin","metformin","empagliflozin","liraglutide","semaglutide",
    "omeprazole","pantoprazole","ondansetron","morphine","oxycodone","acetaminophen",
    "nitroglycerin","norepinephrine","sacubitril","spironolactone",
]
PROCEDURE_PATTERNS = [
    "coronary artery bypass grafting","CABG","percutaneous coronary intervention","PCI",
    "cardiac catheterization","echocardiogram","ECHO","transesophageal echocardiogram",
    "CT chest","CT abdomen","CT head","CXR","chest X-ray","MRI brain",
    "bronchoscopy","thoracentesis","paracentesis","lumbar puncture",
    "colonoscopy","endoscopy","mechanical ventilation","intubation","hemodialysis",
]
ANATOMY_PATTERNS = [
    "right lower lobe","left lower lobe","right upper lobe","left upper lobe",
    "sternotomy site","thoracotomy site","left ventricle","right ventricle",
    "mitral valve","aortic valve","aorta","pulmonary artery",
    "liver","spleen","kidney","gallbladder","pancreas","colon",
]
SYMPTOM_PATTERNS = [
    "chest pain","dyspnea","shortness of breath","SOB","cough","productive cough",
    "fever","chills","nausea","vomiting","abdominal pain","headache",
    "dizziness","syncope","palpitations","fatigue","weakness","edema",
    "hemoptysis","confusion","altered mental status",
]
FINDING_PATTERNS = [
    "bilateral pitting edema","pitting edema","S3 gallop","S4 gallop","murmur",
    "crackles","rales","wheezes","rhonchi","decreased breath sounds",
    "consolidation","air bronchograms","ground glass opacity","atelectasis",
    "ST elevation","ST depression","T-wave inversion","pulmonary nodule",
    "right lower lobe consolidation","bilateral pleural effusions","cardiomegaly",
    "right ventricular enlargement","hepatomegaly","splenomegaly","jaundice",
]

ENTITY_LISTS = {
    "DISEASE":DISEASE_PATTERNS,"DRUG":DRUG_PATTERNS,"PROCEDURE":PROCEDURE_PATTERNS,
    "ANATOMY":ANATOMY_PATTERNS,"SYMPTOM":SYMPTOM_PATTERNS,"FINDING":FINDING_PATTERNS,
}
total = sum(len(v) for v in ENTITY_LISTS.values())
print(f"Total clinical patterns: {total}")
for label, patterns in ENTITY_LISTS.items():
    print(f"  {label:12s}: {len(patterns):3d} patterns")


## 3. spaCy EntityRuler Pipeline

In [ ]:
if SPACY_OK:
    from spacy.pipeline import EntityRuler
    clinical_nlp = spacy.blank("en")
    clinical_nlp.add_pipe("sentencizer")
    ruler = clinical_nlp.add_pipe("entity_ruler", config={"overwrite_ents":True})
    ruler_patterns = []
    for label, patterns in ENTITY_LISTS.items():
        for p in patterns:
            ruler_patterns.append({"label":label,"pattern":p})
            ruler_patterns.append({"label":label,"pattern":p.lower()})
            ruler_patterns.append({"label":label,"pattern":p.title()})
    ruler.add_patterns(ruler_patterns)
    print(f"EntityRuler: {len(ruler_patterns)} patterns loaded")

    doc = clinical_nlp(ANNOTATED[0]["text"])
    print(f"\nNER on T001 ({len(list(doc.ents))} entities):")
    for ent in doc.ents:
        print(f"  [{ent.label_:12s}] '{ent.text}'")


## 4. Regex Fallback NER

In [ ]:
def regex_ner(text, entity_lists):
    entities = []
    for label, patterns in entity_lists.items():
        for pattern in patterns:
            regex = r'\b' + re.escape(pattern) + r'\b'
            for m in re.finditer(regex, text, re.IGNORECASE):
                entities.append({"text":m.group(),"label":label,
                                 "start":m.start(),"end":m.end()})
    # Remove overlapping — keep longer match
    entities.sort(key=lambda x: (x["start"], -(x["end"]-x["start"])))
    filtered = []; prev_end = -1
    for ent in entities:
        if ent["start"] >= prev_end:
            filtered.append(ent); prev_end = ent["end"]
    return filtered

print("Regex NER across corpus:\n")
for ann in ANNOTATED:
    ents = regex_ner(ann["text"], ENTITY_LISTS)
    print(f"{ann['id']}: {len(ents)} entities")
    for e in ents[:4]:
        print(f"  [{e['label']:12s}] '{e['text']}'")
    if len(ents) > 4: print(f"  ... +{len(ents)-4} more")
    print()


## 5. Negation-Aware NER

In [ ]:
NEGATION_TRIGGERS = ["no ","not ","denies ","denied ","without ","w/o ",
                      "never ","absent ","negative ","no evidence of ","rules out "]

def ner_with_negation(text, entity_lists):
    entities = []
    for label, patterns in entity_lists.items():
        for pattern in patterns:
            regex = r'\b' + re.escape(pattern) + r'\b'
            for m in re.finditer(regex, text, re.IGNORECASE):
                context = text[max(0,m.start()-70):m.start()].lower()
                negated = any(neg in context for neg in NEGATION_TRIGGERS)
                entities.append({"text":m.group(),"label":label,
                                 "start":m.start(),"end":m.end(),
                                 "negated":negated,"context":context[-40:].strip()})
    return sorted(entities, key=lambda x: x["start"])

print("NER with negation context (T001):")
print(f"{'Entity':30s} {'Type':12s} {'Negated'}")
print("-"*60)
for ent in ner_with_negation(ANNOTATED[0]["text"], ENTITY_LISTS)[:12]:
    flag = "NEGATED" if ent["negated"] else "Affirmed"
    print(f"  {ent['text']:28s} {ent['label']:12s} {flag}")


## 6. NER Evaluation

In [ ]:
def evaluate_ner(gold_list, pred_list):
    from collections import defaultdict
    gold_text = {(g[0].lower(), g[1]) for g in gold_list}
    pred_text = {(p["text"].lower(), p["label"]) for p in pred_list}
    all_types = set(g[1] for g in gold_list) | set(p["label"] for p in pred_list)
    results = {}
    for etype in all_types:
        g = {x for x in gold_text if x[1]==etype}
        p = {x for x in pred_text if x[1]==etype}
        tp = len(g & p); fp = len(p - g); fn = len(g - p)
        prec = tp/(tp+fp) if tp+fp>0 else 0
        rec  = tp/(tp+fn) if tp+fn>0 else 0
        f1   = 2*prec*rec/(prec+rec) if prec+rec>0 else 0
        results[etype] = {"TP":tp,"FP":fp,"FN":fn,
                          "Precision":round(prec,3),"Recall":round(rec,3),"F1":round(f1,3)}
    return results

all_gold, all_pred = [], []
for ann in ANNOTATED:
    preds = ner_with_negation(ann["text"], ENTITY_LISTS)
    all_gold.extend(ann["gold"])
    all_pred.extend(preds)

eval_df = pd.DataFrame(evaluate_ner(all_gold, all_pred)).T
print("NER Evaluation (text-level match):")
display(eval_df[["Precision","Recall","F1","TP","FP","FN"]])
print(f"\nMacro F1: {eval_df['F1'].mean():.3f}")


In [ ]:
# Entity distribution plot
all_ents = []
for ann in ANNOTATED:
    all_ents.extend(ner_with_negation(ann["text"], ENTITY_LISTS))

counts = pd.Series([e["label"] for e in all_ents]).value_counts()
neg_rates = {t: sum(e["negated"] for e in all_ents if e["label"]==t)/
                max(1,sum(1 for e in all_ents if e["label"]==t))*100
             for t in counts.index}

COLORS = {"DISEASE":"#D65F5F","DRUG":"#4878CF","PROCEDURE":"#6ACC65",
          "ANATOMY":"#B47CC7","SYMPTOM":"#D4A017","FINDING":"#1F7A8C"}

fig, axes = plt.subplots(1,2,figsize=(13,5))
bars = axes[0].bar(counts.index, counts.values,
                   color=[COLORS.get(t,"gray") for t in counts.index], alpha=0.85)
for bar,v in zip(bars,counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2,v+0.1,str(v),ha="center",fontsize=9)
axes[0].set_title("Entity count by type"); axes[0].set_ylabel("Count")

axes[1].bar(neg_rates.keys(), neg_rates.values(),
            color=[COLORS.get(t,"gray") for t in neg_rates.keys()], alpha=0.85)
axes[1].set_title("Negation rate by entity type"); axes[1].set_ylabel("% negated")

plt.tight_layout()
plt.savefig("/tmp/mod05/ner_distribution.png", bbox_inches="tight"); plt.show()


## 7. Knowledge Check
1. Why does EntityRuler have lower recall than a trained NER model?
2. DISEASE recall is 0.72. List three types of disease mentions likely missed.
3. Negation misses 'No ST elevation, but ST depression present' — why?
4. What is the difference between exact span match and token-level evaluation?
5. Add LAB_VALUE extraction: match 'WBC 18.4', 'creatinine 1.8 mg/dL'.

In [ ]:
LAB_VALUE_PAT = (r'(WBC|Hgb|Hemoglobin|Hct|Platelets|PLT|Creatinine|Cr|BUN|Sodium|Na|'
                 r'Potassium|K|Glucose|BNP|Troponin|TSH|INR)'
                 r'\s*:?\s*(\d+\.?\d*)\s*(mg/dL|g/dL|mEq/L|ng/mL|mmol/L|IU/L|%)?')

def extract_lab_values(text):
    labs = []
    for m in re.finditer(LAB_VALUE_PAT, text, re.IGNORECASE):
        labs.append({"analyte":m.group(1),"value":m.group(2),
                     "unit":(m.group(3) or "").strip(),"label":"LAB_VALUE"})
    return labs

for ann in ANNOTATED:
    labs = extract_lab_values(ann["text"])
    if labs:
        print(f"{ann['id']}:")
        for l in labs: print(f"  {l['analyte']:15s} = {l['value']} {l['unit']}")


***
**Next → NB-03: TF-IDF & Clinical Text Classification**
